<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/03-funciona-con-datos-nuevos/clase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Episodio 1.3 · ¿Funciona con datos nuevos?

**Serie:** Del Excel al Machine Learning · **Temporada 1** · 🎬 Video próximamente

## 🎯 Qué vas a aprender

- Por qué un modelo se evalúa con datos que **no vio** al entrenar.
- Separar los datos en **entrenamiento** y **prueba**.
- Medir el error en soles, con el **MAE**.
- Comparar con una **línea base**: lo mínimo que un modelo debe superar.

## 📊 En Excel lo harías así

Armaste una proyección de ventas. ¿Cómo sabes si es buena?

1. Esperas a que pase el mes.
2. En una columna calculas la diferencia sin signo: `=ABS(real - proyectado)`.
3. Sacas el `=PROMEDIO()` (`AVERAGE`) de esas diferencias.

Hoy hacemos algo parecido, sin esperar: guardamos filas que el modelo no ve.

## 🐍 En Python

### Paso 1: preparar los datos

Usamos la tabla del episodio anterior. Traemos dos herramientas nuevas de scikit-learn, que usaremos enseguida.

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
def por_cliente_y_mes(ventas):
    """Una fila por cliente y mes en que compró, con sus líneas de pedido y su monto."""
    meses = ventas["fecha"].str[:7]
    return (
        ventas.groupby(["id_cliente", meses])
        .agg(lineas=("total", "size"), total=("total", "sum"))
        .reset_index()
    )

In [3]:
ventas = pd.read_csv(DATOS + "ventas.csv")

mensual = por_cliente_y_mes(ventas)

x = mensual[["lineas"]]
y = mensual["total"]

len(mensual)

6459

### Paso 2: el problema de evaluar con lo mismo

Imagina a un alumno al que le das el examen con las respuestas para estudiar, y luego le tomas ese mismo examen. Sacar 20 no prueba que aprendió.

Con un modelo pasa igual. Por eso guardamos una parte de los datos, y el modelo **no la ve** al entrenar:

- **Entrenamiento:** de aquí aprende.
- **Prueba:** aquí lo evaluamos, con filas que no vio.

`train_test_split()` los separa al azar. `test_size=0.2` guarda el 20 % para la prueba. `random_state=42` fija el azar, para que siempre salga la misma separación. El orden de lo que devuelve importa: primero las dos `x` y luego las dos `y`.

In [4]:
x_entreno, x_prueba, y_entreno, y_prueba = train_test_split(x, y, test_size=0.2, random_state=42)

print(len(x_entreno), len(x_prueba))

5167 1292


### Paso 3: medir el error en soles

Entrenamos **solo** con `x_entreno` y predecimos en `x_prueba`. Calculamos el error como en Excel: la diferencia sin signo, fila por fila, y su promedio.

In [5]:
modelo = LinearRegression()
modelo.fit(x_entreno, y_entreno)

prediccion = modelo.predict(x_prueba)

print((y_prueba - prediccion).abs().mean().round(2))

702.94


Eso es el **error absoluto medio**, o **MAE** por sus siglas en inglés. Se lee en soles: en promedio, la predicción se equivoca en esa cantidad. Es mucho más fácil de explicar a un gerente que el R².

scikit-learn ya trae la función, y da lo mismo:

In [6]:
print(round(mean_absolute_error(y_prueba, prediccion), 2))
print(round(mean_absolute_error(y_entreno, modelo.predict(x_entreno)), 2))

702.94
688.16


`mean_absolute_error()` entrega un número simple, y por eso se redondea con `round()`. La segunda cifra es el error en entrenamiento. Se parecen: una recta tiene solo dos números para más de 5 mil filas, así que no alcanza a "aprenderse las respuestas". Eso no significa que sea buena; con modelos más complejos, la diferencia puede ser enorme, y lo veremos en el episodio 1.6.

### Paso 4: ¿es bueno o malo?

Un error de unos S/ 700, ¿es mucho? Un número solo no dice nada. Hay que compararlo con algo: una **línea base**, la respuesta más simple que se te ocurra.

La más fácil de explicar es predecir para todos el **promedio** de los datos de entrenamiento:

In [7]:
promedio = y_entreno.mean()

print(round(promedio, 2))
print((y_prueba - promedio).abs().mean().round(2))

2087.11
1449.4


La recta se equivoca en menos de la mitad. Ahora sí podemos decir que aporta algo.

> ⚠️ Un modelo que no le gana a su línea base no sirve, por sofisticado que sea. **Compara siempre.**

### Entonces, ¿ya podemos confiar?

Todavía no, por dos razones que ningún cálculo muestra:

- Este modelo usa las líneas de pedido del mes, y esas **no se conocen hasta que termina el mes**. Es como predecir un partido sabiendo cuántos tiros al arco hubo.
- Al separar al azar, mezclamos los meses: entrenamos con diciembre para "predecir" marzo. La vida real solo va hacia adelante.

Las dos se resuelven en el próximo episodio.

## 🤖 Pídeselo a la IA

> *"Mi modelo estima el costo de reparar un vehículo con un error absoluto medio de S/ 90. La reparación típica cuesta S/ 500. ¿Cómo le explico a un gerente si es bueno o malo? ¿Con qué debería compararlo?"*

La IA te ayuda a escribir y entender código, pero **tú debes entender lo que hace**. Revisa cada respuesta antes de usarla, y no pegues datos de tu empresa.

## ✅ Resumen

| En Excel | En Python |
|---|---|
| Separar una parte de las filas para comprobar | `train_test_split(x, y, test_size=0.2, random_state=42)` |
| `=ABS(real - proyectado)` | `(y_prueba - prediccion).abs()` |
| `=PROMEDIO()` (`AVERAGE`) de esas diferencias | `mean_absolute_error(y_prueba, prediccion)` |
| Comparar con el promedio | Línea base: `y_entreno.mean()` |

## 📝 Ahora te toca

Abre [el ejercicio en Colab](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/03-funciona-con-datos-nuevos/ejercicio.ipynb) y evalúa tus modelos. Cuando termines, compara con [la solución](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/03-funciona-con-datos-nuevos/solucion.ipynb).

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0